In [11]:
import torch
import numpy as np
import pandas as pd
import random
import os
from collections import Counter
from sklearn import tree as sktree
from sklearn.cluster import DBSCAN
from pm4py import save_vis_petri_net
from lark import Tree, Token

from config import DATA_DIR
from core import *
from utils import *
import matplotlib.pyplot as plt

NARY = 1
PROBABILITIES = 0.5,0,0.5,0 # XOR, PAR, SEQ, LOOP
COVERAGE = 1  # Frazione di regioni XOR/loop classificate (0.0 = tutto random, 1.0 = tutto classificato)
LOOP = True # Creiamo i classificatori per i loop?
XOR = True # Creiamo i classificatori per gli xor?
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [12]:
iterations = 3  # Quante iterazioni diverse (numero regioni prima di minimizzare in teoria)
current_string = SEED_STRING
for _ in range(iterations):
    current_string = replace_random_underscore(current_string, PROBABILITIES)

process = replace_underscores(current_string)
tree = PARSER.parse(process)

complexity_nested, complexity_parallel = compute_process_complexity(tree)

'''tree = Tree('loop', [
        Tree('sequential', [
            Tree('loop', [Tree('task', [Token('NAME', 'T1')])]),
            Tree('task', [Token('NAME', 'T2')]),
            Tree('loop', [Tree('task', [Token('NAME', 'T3')])]),
            Tree('loop', [Tree('sequential', [Tree('loop', [Tree('task', [Token('NAME', 'T4')])]), Tree('task', [Token('NAME', 'T5')])])])
        ])
    ])'''

if NARY:
    tree = createNAryTree(tree)

print(tree)
print(complexity_nested)
print(complexity_parallel)

Tree('sequential', [Tree('xor', [Tree('sequential', [Tree('task', [Token('NAME', 'T1')]), Tree('task', [Token('NAME', 'T2')])]), Tree('task', [Token('NAME', 'T3')])]), Tree('task', [Token('NAME', 'T4')])])
1
1


In [13]:
# Oggetto PetriNetP e Generator

net = PetriNetP(tree)
generator = Generator(1, net)
generator.generateTrace(False)

[['start_X1', 'start_T3', 'end_T3', 'end_X1', 'start_T4', 'end_T4']]

In [16]:
save_vis_petri_net(
    net.net,
    net.initial_marking,
    net.final_marking,
    "bpmn.png",
    format="png"
)

ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [ ]:
# Creazione dei decision tree per gli xor
classifier_dict_xor = {}
if XOR:
    xors_to_classify = random.sample(net.xor_regions, round(COVERAGE * len(net.xor_regions)))

    for xor in xors_to_classify:
        result = create_xor_data(xor, generator.generatedTraces, net.net)
        if result is None:
            continue

        x, y, dict_loop_step_encoding, max_len, class_to_branch = result

        dt = sktree.DecisionTreeClassifier(max_depth=5, class_weight='balanced') #Se mettiamo class_weight='balanced' ﷿﷿ come se avessimo pompato le tracce
        dt.fit(x, y)
        classifier_dict_xor[xor] = (dt, dict_loop_step_encoding, max_len, class_to_branch)

    print(f'XOR classificati: {list(classifier_dict_xor.keys())} / {net.xor_regions}')
    print(classifier_dict_xor)

In [ ]:
# Creazione dei decision tree per 'limitare' i loop e provare a dargli un 'senso'
classifier_dict_loop = {}
if LOOP:
    loops_to_classify = random.sample(net.loop_regions, round(COVERAGE * len(net.loop_regions)))

    for loop in loops_to_classify:
        result = create_loop_data(loop, generator.generatedTraces)
        if result is None:
            continue

        x, y, dict_loop_step_encoding, max_len = result

        dt = sktree.DecisionTreeClassifier(max_depth=5, class_weight='balanced') #Se mettiamo class_weight='balanced' ﷿﷿ come se avessimo pompato le tracce
        dt.fit(x, y)
        classifier_dict_loop[loop] = (dt, dict_loop_step_encoding, max_len)

    print(f'Loop classificati: {list(classifier_dict_loop.keys())} / {net.loop_regions}')
    print(classifier_dict_loop)

In [ ]:
for clf,_,_,ctb in classifier_dict_xor.values(): #ctb -> class to branch
    plt.figure(figsize=(10, 8))
    sktree.plot_tree(clf,
               class_names=[str(i) for i in range(len(ctb.keys()))])
    plt.show()

In [ ]:
for clf,_,_ in classifier_dict_loop.values():
    plt.figure(figsize=(10, 8))
    sktree.plot_tree(clf,
               class_names=["0", "1"])
    plt.show()

In [ ]:
# Rigenero le tracce utilizzando i classificatori
generator.generateTraceCond(classifier_dict_loop, classifier_dict_xor)

In [ ]:
# Creazione matrice identit﷿﷿ delle regioni
df_region_identity = pd.DataFrame.from_dict(net.node_identity, orient='index').sort_index()
df_region_identity.columns = ['X', '+', '->', '<>']
print(df_region_identity)

# Creazione matrice regioni-figli per le regioni
df_region_children = pd.Series(net.node_children).explode()
df_region_children = pd.crosstab(df_region_children.index, df_region_children)
df_region_children = df_region_children.reindex(index=net.regions, columns=net.regions + net.tasks, fill_value=0)
df_region_children = df_region_children.astype(int)
df_region_children.index.name = None
df_region_children.columns.name = None
print(df_region_children)

# Codifica delle tracce generate
traceEncoded_regions, traceEncoded_tasks = get_encoding(
    generator.generatedTraces, net.regions, net.tasks, net.open_clauses, net.end_clauses
)

# Trovo numero regioni e numero task effettivo
num_regions = len([i for i in traceEncoded_regions.index if str(i).startswith('R')])
num_tasks = len([i for i in traceEncoded_tasks.index   if str(i).startswith('T')])

# Creo il dataframe unico (regioni + task)
df_traces_complete = pd.concat([traceEncoded_regions, traceEncoded_tasks], axis=0)
print(f'num_regions={num_regions}, num_tasks={num_tasks}, shape={df_traces_complete.shape}')

In [ ]:
from joblib import Parallel, delayed

all_traces     = getall_traces(num_regions + num_tasks, df_traces_complete)
trace_counts   = Counter(all_traces)
traces_encoded = list(trace_counts.keys())
trace_weights  = list(trace_counts.values())
n_unique       = len(traces_encoded)
num_features   = num_regions + num_tasks

DBSCAN_THRESHOLD = 1000  # sotto ﷿﷿﷿ DBSCAN esatto, sopra ﷿﷿﷿ CLARA K-Medoids
CLARA_M          = 1000  # dimensione campione CLARA
CLARA_N_WORKERS  = 12    # core paralleli
MIN_CLUSTERS     = 5     # k per CLARA; per DBSCAN usa find_epsilon o eps manuale

print(f"Tracce uniche: {n_unique}  ﷿﷿﷿  {'DBSCAN' if n_unique <= DBSCAN_THRESHOLD else 'CLARA K-Medoids'}")

In [ ]:
def _compute_nearest(trace, medoid_traces, num_features):
    return int(np.argmin([
        edit_distance_weighted_levenshtein(trace, m, num_features, num_features, hamming_distance)
        for m in medoid_traces
    ]))

def _pam_on_sample(dist_matrix, k):
    M = len(dist_matrix)
    medoids     = [int(np.argmin(dist_matrix.sum(axis=1)))]
    non_medoids = [i for i in range(M) if i != medoids[0]]
    for _ in range(k - 1):
        min_dists = dist_matrix[:, medoids].min(axis=1)
        best_gain, best_o = -np.inf, None
        for o in non_medoids:
            gain = float(np.maximum(0.0, min_dists - dist_matrix[:, o]).sum())
            if gain > best_gain:
                best_gain, best_o = gain, o
        medoids.append(best_o)
        non_medoids.remove(best_o)
    improved = True
    while improved:
        improved     = False
        current_cost = dist_matrix[:, medoids].min(axis=1).sum()
        for mi in range(k):
            for o in [x for x in range(M) if x not in medoids]:
                trial     = medoids[:]
                trial[mi] = o
                new_cost  = dist_matrix[:, trial].min(axis=1).sum()
                if new_cost < current_cost - 1e-9:
                    medoids, current_cost, improved = trial, new_cost, True
                    break
            if improved:
                break
    return medoids

if n_unique <= DBSCAN_THRESHOLD:
    distance_matrix = create_distance_matrix(n_unique, traces_encoded, num_features, n_workers=CLARA_N_WORKERS)
    eps = 25  # modifica manualmente, oppure: find_epsilon(distance_matrix, trace_weights, MIN_CLUSTERS)
    dbscan = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    dbscan.fit(distance_matrix, sample_weight=trace_weights)
    cluster_labels = dbscan.labels_
    print(f"DBSCAN: {len(set(cluster_labels))} cluster (eps={eps}, n_unique={n_unique})")
else:
    M_actual      = min(CLARA_M, n_unique)
    sample_idx    = np.random.choice(n_unique, size=M_actual, replace=False)
    sample_traces = [traces_encoded[i] for i in sample_idx]
    dist_sample   = create_distance_matrix(M_actual, sample_traces, num_features, n_workers=CLARA_N_WORKERS)
    medoid_idx    = _pam_on_sample(dist_sample, MIN_CLUSTERS)
    medoid_traces = [sample_traces[i] for i in medoid_idx]
    cluster_labels = np.array(
        Parallel(n_jobs=CLARA_N_WORKERS)(
            delayed(_compute_nearest)(t, medoid_traces, num_features)
            for t in traces_encoded
        )
    )
    print(f"CLARA: {MIN_CLUSTERS} cluster, M={M_actual}, n_unique={n_unique}")

pd.DataFrame({
    'Trace_Type': [str(t) for t in traces_encoded],
    'Frequency':  trace_weights,
    'N_Cluster':  cluster_labels,
}).sort_values(by=['N_Cluster', 'Frequency'], ascending=[True, False])

In [ ]:
'''Prendo tot tracce per cluster --> per bilanciare le tracce generate e per non avere prevalentemente tracce dello stesso cluster'''

num_trace_per_cluster = 1000
balanced_traces, balanced_columns = get_balance_traces_by_cluster(
    traces_encoded, cluster_labels, num_trace_per_cluster
)

df_traces_balanced = pd.DataFrame(balanced_columns).T
df_traces_balanced.index = df_traces_complete.index
print(df_traces_balanced)

In [ ]:
# Prendo i rispettivi dataframe per le regioni e per le task
df_regions = df_traces_balanced.head(num_regions).copy()
df_tasks   = df_traces_balanced.tail(num_tasks).copy()

df_traces_balanced_T = df_traces_balanced.T.copy()

# Vocabolario completo (regioni + task)
unique_cols_complete = df_traces_balanced_T.drop_duplicates()
unique_tup_complete  = [tuple(x) for x in unique_cols_complete.values]
bit_to_id_complete   = {v: i for i, v in enumerate(unique_tup_complete)}
id_to_bit_complete   = {i: v for i, v in enumerate(unique_tup_complete)}
vocab_size_complete  = len(unique_cols_complete)
encode_complete      = lambda a: [bit_to_id_complete[tuple(x)] for x in a]
decode_complete      = lambda b: [id_to_bit_complete[x] for x in b]

# Vocabolario regioni
df_regions_T        = df_regions.T
unique_cols_regions = df_regions_T.drop_duplicates()
unique_tup_regions  = [tuple(x) for x in unique_cols_regions.values]
bit_to_id_regions   = {v: i for i, v in enumerate(unique_tup_regions)}
id_to_bit_regions   = {i: v for i, v in enumerate(unique_tup_regions)}
vocab_size_regions  = len(unique_cols_regions)
encode_regions      = lambda a: [bit_to_id_regions[tuple(x)] for x in a]
decode_regions      = lambda b: [id_to_bit_regions[x] for x in b]

# Vocabolario task
df_tasks_T        = df_tasks.T
unique_cols_tasks = df_tasks_T.drop_duplicates()
unique_tup_tasks  = [tuple(x) for x in unique_cols_tasks.values]
bit_to_id_tasks   = {v: i for i, v in enumerate(unique_tup_tasks)}
id_to_bit_tasks   = {i: v for i, v in enumerate(unique_tup_tasks)}
vocab_size_tasks  = len(unique_cols_tasks)
encode_tasks      = lambda a: [bit_to_id_tasks[tuple(x)] for x in a]
decode_tasks      = lambda b: [id_to_bit_tasks[x] for x in b]

print(f'vocab complete={vocab_size_complete}, regions={vocab_size_regions}, tasks={vocab_size_tasks}')

In [ ]:
# Decodifico le tracce bilanciate con la funzione get_decoding
traces_balanced_decoded = get_decoding(balanced_traces, net.regions, net.tasks)

possible_tasks = [] # Per ogni task dobbiamo considerare sia start_'task' sia end_'task' --> per la generazione dei tempi
for task in net.tasks:
    possible_tasks.append(f'start_{task}')
    possible_tasks.append(f'end_{task}')

k_partition = 3
max_depth = 5

# Seleziono k-pattern per differenziare i tempi delta dei vari step (ex: in modo che start_T1 non abbia sempre 4)
tree_times_task_dict = {}
for task in possible_tasks:
    miner = TracePatternMiner()
    miner.root.name = task # Sovrascrivo il nome della radice (sar﷿﷿ la parte di task che sto usando)

    num_traces_for_task = 0
    for trace in traces_balanced_decoded:
        for idx in [i for i, step in enumerate(trace) if step == task]: # TUTTE le occorrenze (non solo la prima: serve per i loop)
            subtrace_task = trace[:idx] # Prendo la traccia fino al punto del task (task escluso)
            reversed_subtrace = list(reversed(subtrace_task)) # Giro la traccia presa --> la considero al contrario
            miner.fit_trace(reversed_subtrace, max_depth) # Vado a conteggiarla nel miner
            num_traces_for_task += 1

    # or len(miner.nodes) == 0
    if num_traces_for_task == 0 or len(miner.nodes) == 0: # Se non ho trovato nemmeno una partecipazione di quella task (in teoria NON dovrebbe capitare)
        tree_times_task_dict[task] = {
            'partitions': {'RESIDUALS': {'numTraces': num_traces_for_task, 'nodes': []}},
            'times_map':  {'RESIDUALS': 0.0} # Se non ha storia, il tempo normalizzato ﷿﷿ sempre il minimo (0.0)
        }
        continue

    partitions = miner.select_patterns(num_traces_for_task, k_partition) # Cerco i pattern
    times_map = miner.create_normalized_time_map(partitions) # Creo il mapping time partitions --> assegno un delta ad ogni partizione (con funzione esponenziale)
    tree_times_task_dict[task] = {'partitions': partitions, 'times_map': times_map}
    print(task)
    print(times_map)

In [ ]:
# A questo punto assegno i tempi alle tracce (basandomi sui pattern che ho gi﷿﷿ trovato in precedenza)
times = []
trace_times_list = []
window = 5

for trace in traces_balanced_decoded: # Assegno i tempi alle tracce
    this_trace_times = []
    for i, element in enumerate(trace):
        if i == 0:
            this_trace_times.append(0.0)
            times.append(0.0)
            continue
        times_map = tree_times_task_dict[element]['times_map']
        generated_time = assign_time(trace[:i], times_map, window)
        this_trace_times.append(generated_time)
        times.append(generated_time)

    print(this_trace_times)
    trace_times_list.append(this_trace_times)

print(f'Time steps totali: {len(times)}')

In [ ]:
all_unique_steps = set(e for trace in traces_balanced_decoded for e in trace)
all_unique_steps.add('PAD') # Aggiungiamo il carattere speciale
dict_task_step_encoding = {task: i for i, task in enumerate(all_unique_steps)}

classifier_dict_tasks = {}

for task in possible_tasks:
    result = create_task_data(task, traces_balanced_decoded, trace_times_list, dict_task_step_encoding)

    if result is None:
        continue

    x, y, max_len = result

    rt = sktree.DecisionTreeRegressor(max_depth=5)
    rt.fit(x, y)

    classifier_dict_tasks[task] = (rt, max_len)
    print(f'{task} | R﷿﷿={rt.score(x, y):.4f}')

In [ ]:
data_complete = torch.tensor(encode_complete(df_traces_balanced_T.values), dtype=torch.long)
data_regions  = torch.tensor(encode_regions(df_regions_T.values),          dtype=torch.long)
data_tasks    = torch.tensor(encode_tasks(df_tasks_T.values),               dtype=torch.long)
data_times    = torch.tensor(times,                                          dtype=torch.float)

n = int(0.8 * len(df_traces_balanced_T))
print(f'Train={n}, Val={len(df_traces_balanced_T)-n}')
print(f'data_complete={data_complete.shape}, data_regions={data_regions.shape}, data_tasks={data_tasks.shape}, data_times={data_times.shape}')

In [ ]:
torch.save({
    # Tensori per il training
    'data_complete':      data_complete,
    'data_regions':       data_regions,
    'data_tasks':         data_tasks,
    'data_times':         data_times,
    'n':                  n,
    # Dimensioni vocabolari
    'vocab_size_complete': vocab_size_complete,
    'vocab_size_regions':  vocab_size_regions,
    'vocab_size_tasks':    vocab_size_tasks,
    'num_regions':         num_regions,
    'num_tasks':           num_tasks,
    # Dizionari per encode/decode (lambdas non picklabili, si ricostruiscono da questi)
    'bit_to_id_complete': bit_to_id_complete,
    'id_to_bit_complete': id_to_bit_complete,
    'bit_to_id_regions':  bit_to_id_regions,
    'id_to_bit_regions':  id_to_bit_regions,
    'bit_to_id_tasks':    bit_to_id_tasks,
    'id_to_bit_tasks':    id_to_bit_tasks,
    # Metadati processo
    "net":                net,
    'regions':            net.regions,
    'tasks':              net.tasks,
    # Classifiers
    'classifier_dict_tasks':   classifier_dict_tasks,
    'dict_task_step_encoding': dict_task_step_encoding,
}, DATA_DIR / 'prepared_data.pt')

print(f'Salvato in {DATA_DIR / "prepared_data.pt"}')